In [71]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
import numpy as np
from scipy import stats

train = pd.read_csv('train.csv')
eval_data = pd.read_csv('eval.csv')

In [ ]:

features = [
    'winperc_avg3',    # R²=0.31  Best overall predictor, stable 3yr average
    'scgf_p_avg3',     # R²=0.28  Scoring-chance goal %, finishing quality
    'x_gf_p_lag1',     # R²=0.28  Expected goal share, process metric
    'hdgf_p_avg3',     # R²=0.24  High-danger goal %, chance quality
    'ga_avg3',         # R²=0.15  Goals allowed, defensive quality
    'sv_avg3',         # R²=0.05  Save %, goaltending skill
    'pdo_lag1',        # R²=0.12  Luck indicator, regression flag
    'ldsf_p_lag1',     # R²=0.14  Low-danger shot share, territorial control
]

In [73]:
# setting my independant and dependant variabels 

X = train[features]
y = train['winperc']
X_eval = eval_data[features]


In [74]:
all_features = [col for col in train.columns if col not in ['team', 'season', 'winperc']]

rows = []
for feature in all_features:
    x = train[feature]
    y = train['winperc']
    
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    
    rows.append({
        'Feature': feature,
        'Correlation': round(r_value, 4),
        'R-Squared': round(r_value**2, 4),
        'P-Value': round(p_value, 4),
        'Significant': 'Yes' if p_value < 0.05 else 'No'
    })

table = pd.DataFrame(rows)
table = table.sort_values('R-Squared', ascending=False)
print(table.to_string(index=False))

     Feature  Correlation  R-Squared  P-Value Significant
winperc_lag1       0.5698     0.3247   0.0000         Yes
point_p_lag1       0.5626     0.3165   0.0000         Yes
winperc_avg3       0.5595     0.3131   0.0000         Yes
   gf_p_lag1       0.5596     0.3131   0.0000         Yes
point_p_avg3       0.5520     0.3047   0.0000         Yes
 scgf_p_lag1       0.5375     0.2889   0.0000         Yes
 scgf_p_avg3       0.5328     0.2839   0.0000         Yes
   gf_p_avg3       0.5281     0.2789   0.0000         Yes
    row_avg3       0.5269     0.2776   0.0000         Yes
 x_gf_p_lag1       0.5249     0.2756   0.0000         Yes
  scf_p_lag1       0.5231     0.2736   0.0000         Yes
 scsf_p_lag1       0.5226     0.2731   0.0000         Yes
      w_avg3       0.5046     0.2546   0.0000         Yes
      l_avg3      -0.5005     0.2505   0.0000         Yes
   sf_p_lag1       0.4985     0.2485   0.0000         Yes
 scsf_p_avg3       0.4981     0.2481   0.0000         Yes
 hdgf_p_avg3  

In [75]:
# splitting up my training data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)



In [76]:
# Fitting my model - tuned to prevent overfitting on 300 rows
model = DecisionTreeRegressor(max_depth=3, min_samples_leaf=8, min_samples_split=5, random_state=42)
model.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",3
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",8
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf

In [77]:
# Making the predictions on my training data 
val_predictions = model.predict(X_val)


In [78]:
rmse = np.sqrt(mean_squared_error(y_val, val_predictions))
print(f'Validation RMSE: {rmse:.4f}')

Validation RMSE: 0.0839


In [79]:
# 5-fold cross-validation for a more reliable RMSE estimate
# (single train/test split depends on which 20% was held out)
cv_scores = cross_val_score(
    DecisionTreeRegressor(max_depth=3, min_samples_leaf=8, min_samples_split=5, random_state=42),
    X, y, cv=5, scoring='neg_mean_squared_error'
)
cv_rmse = np.sqrt(-cv_scores.mean())
print(f'5-Fold CV RMSE: {cv_rmse:.4f}')

5-Fold CV RMSE: 0.0889
